In [ ]:
#hide
! [ -e /content ] && pip install -Uqq fastbook
import fastbook
fastbook.setup_book()

In [ ]:
#hide
from fastbook import *

# Approfondimento sulle architetture applicative

Ora ci troviamo nella entusiasmante posizione in cui possiamo comprendere appieno le architetture che utilizziamo per i nostri modelli all'avanguardia per la visione artificiale, l'elaborazione del linguaggio naturale e l'analisi tabulare. In questo capitolo inseriremo tutti i dettagli mancanti su come funzionano i modelli applicativi di Fastai e ti mostreremo come costruire i modelli che utilizzano.

Torneremo anche alla pipeline di preelaborazione dei dati personalizzata che abbiamo visto in <<chapter_midlevel_data>> per le reti siamesi e ti mostreremo come utilizzare i componenti nella libreria fastai per creare modelli preaddestrati personalizzati per nuove attività.

Inizieremo con la visione artificiale.

##Visione artificiale

Per le applicazioni di visione artificiale utilizziamo le funzioni "vision_learner" e "unet_learner" per costruire i nostri modelli, a seconda dell'attività. In questa sezione esploreremo come costruire gli oggetti "Learner" che abbiamo utilizzato nelle Parti 1 e 2 di questo libro.

### vision_learner

Diamo un'occhiata a cosa succede quando utilizziamo la funzione `vision_learner`. Iniziamo passando a questa funzione un'architettura da utilizzare per il *corpo* della rete. La maggior parte delle volte utilizziamo una ResNet, che sai già come creare, quindi non abbiamo bisogno di approfondire ulteriormente l'argomento. I pesi preaddestrati vengono scaricati secondo necessità e caricati in ResNet.

Quindi, per il trasferimento dell'apprendimento, la rete deve essere *tagliata*. Ciò si riferisce all'eliminazione dello strato finale, che è responsabile solo della categorizzazione specifica di ImageNet. In effetti, non eliminiamo solo questo livello, ma tutto, dal livello di pooling medio adattivo in poi. La ragione di ciò diventerà chiara tra un attimo. Poiché architetture diverse potrebbero utilizzare diversi tipi di livelli di pooling, o anche tipi di *teste* completamente diversi, non cerchiamo solo il livello di pooling adattivo per decidere dove tagliare il modello pre-addestrato. Abbiamo invece un dizionario di informazioni che viene utilizzato per ciascun modello per determinare dove finisce il suo corpo e inizia la sua testa. Lo chiamiamo "model_meta": eccolo per resnet-50:

In [ ]:
model_meta[resnet50]

{'cut': -2,
 'split': <function fastai.vision.learner._resnet_split(m)>,
 'stats': ([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])}

> gergo: Corpo e Testa: La "testa" di una rete neurale è la parte specializzata per un compito particolare. Per una CNN, è generalmente la parte dopo il livello di pooling medio adattivo. Il "corpo" è tutto il resto e include la "radice" (di cui abbiamo appreso in <<chapter_resnet>>).

Se prendiamo tutti gli strati prima del punto di taglio `-2`, otteniamo la parte del modello che fastai manterrà per l'apprendimento del trasferimento. Adesso indossiamo la nostra nuova testa. Questo viene creato utilizzando la funzione "create_head":

In [ ]:
#hide_output
create_head(20,2)

Sequential(
  (0): AdaptiveConcatPool2d(
    (ap): AdaptiveAvgPool2d(output_size=1)
    (mp): AdaptiveMaxPool2d(output_size=1)
  )
  (1): full: False
  (2): BatchNorm1d(20, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (3): Dropout(p=0.25, inplace=False)
  (4): Linear(in_features=20, out_features=512, bias=False)
  (5): ReLU(inplace=True)
  (6): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (7): Dropout(p=0.5, inplace=False)
  (8): Linear(in_features=512, out_features=2, bias=False)
)

```
Sequenziale(
  (0): AdattivoConcatPool2d(
    (ap): AdaptiveAvgPool2d(output_size=1)
    (mp): AdaptiveMaxPool2d(output_size=1)
  )
  (1): Appiattisci()
  (2): BatchNorm1d(20, eps=1e-05, quantità di moto=0,1, affine=Vero)
  (3): Dropout(p=0,25, inplace=Falso)
  (4): Lineare(in_features=20, out_features=512, bias=False)
  (5): ReLU(inplace=True)
  (6): BatchNorm1d(512, eps=1e-05, quantità di moto=0,1, affine=Vero)
  (7): Dropout(p=0,5, inplace=Falso)
  (8): Lineare(in_features=512, out_features=2, bias=False)
)
```

Con questa funzione puoi scegliere quanti layer lineari aggiuntivi aggiungere alla fine, quanto dropout utilizzare dopo ciascuno di essi e che tipo di pooling utilizzare. Per impostazione predefinita, fastai applicherà sia il pooling medio che quello massimo e concatenerà i due insieme (questo è il livello `AdaptiveConcatPool2d`). Questo non è un approccio particolarmente comune, ma è stato sviluppato in modo indipendente presso Fastai e altri laboratori di ricerca negli ultimi anni e tende a fornire qualche piccolo miglioramento rispetto all'utilizzo del pooling medio.

fastai è leggermente diverso dalla maggior parte delle librerie in quanto per impostazione predefinita aggiunge due livelli lineari, anziché uno, nella testata della CNN. La ragione di ciò è che il transfer learning può comunque essere utile anche, come abbiamo visto, quando si trasferisce il modello preaddestrato a domini molto diversi. Tuttavia, in questi casi, l’utilizzo di un solo strato lineare difficilmente è sufficiente; abbiamo scoperto che l'utilizzo di due livelli lineari può consentire di utilizzare l'apprendimento del trasferimento più rapidamente e facilmente, in più situazioni.

> nota: One Last Batchnorm?: Un parametro di `create_head` che vale la pena esaminare è `bn_final`. Impostandolo su "true" verrà aggiunto un livello batchnorm come livello finale. Ciò può essere utile per aiutare il modello a ridimensionarsi in modo appropriato per le attivazioni di output. Non abbiamo ancora visto questo approccio pubblicato da nessuna parte, ma abbiamo scoperto che funziona bene nella pratica ovunque lo abbiamo utilizzato.

Diamo ora un'occhiata a cosa ha fatto `unet_learner` nel problema di segmentazione che abbiamo mostrato in <<chapter_intro>>.

### unet_studente

Una delle architetture più interessanti nel deep learning è quella che abbiamo utilizzato per la segmentazione in <<chapter_intro>>. La segmentazione è un compito impegnativo, perché l'output richiesto è in realtà un'immagine, o una griglia di pixel, contenente l'etichetta prevista per ogni pixel. Ci sono altre attività che condividono un design di base simile, come aumentare la risoluzione di un'immagine (*super-risoluzione*), aggiungere colore a un'immagine in bianco e nero (*colorazione*) o convertire una foto in un dipinto sintetico (*trasferimento di stile*): queste attività sono trattate in un capitolo [online](https://book.fast.ai/) di questo libro, quindi assicurati di controllarlo dopo aver letto questo capitolo. In ogni caso, iniziamo con un'immagine e la convertiamo in un'altra immagine delle stesse dimensioni o proporzioni, ma con i pixel alterati in qualche modo. Li chiamiamo *modelli di visione generativa*.

Il modo in cui lo facciamo è iniziare con lo stesso identico approccio allo sviluppo di una testata della CNN come abbiamo visto nel problema precedente. Iniziamo con ResNet, ad esempio, e tagliamo il livello di pooling adattivo e tutto il resto. Quindi sostituiamo questi livelli con la nostra testa personalizzata, che svolge il compito generativo.

C'erano molti gesti in quell'ultima frase! Come diavolo possiamo creare una testata della CNN che generi un'immagine? Se iniziamo, ad esempio, con un'immagine di input da 224 pixel, alla fine del corpo ResNet avremo una griglia 7×7 di attivazioni convoluzionali. Come possiamo convertirlo in una maschera di segmentazione da 224 pixel?

Naturalmente lo facciamo con una rete neurale! Quindi abbiamo bisogno di una sorta di livello che possa aumentare la dimensione della griglia in una CNN. Un approccio molto semplice a questo problema consiste nel sostituire ogni pixel nella griglia 7×7 con quattro pixel in un quadrato 2×2. Ciascuno di questi quattro pixel avrà lo stesso valore: questo è noto come *interpolazione del vicino più vicino*. PyTorch fornisce un livello che fa questo per noi, quindi un'opzione è creare una testa che contenga livelli convoluzionali stride-1 (insieme ai livelli batchnorm e ReLU come al solito) intervallati da livelli di interpolazione del vicino più vicino 2×2. In effetti, puoi provarlo ora! Vedi se riesci a creare una testa personalizzata progettata in questo modo e provala nell'attività di segmentazione di CamVid. Dovresti scoprire che ottieni dei risultati ragionevoli, anche se non saranno buoni come i nostri risultati <<chapter_intro>>.

Un altro approccio consiste nel sostituire la combinazione del vicino più vicino e della convoluzione con una *convoluzione trasposta*, altrimenti nota come *mezza convoluzione del passo*. Questa è identica a una convoluzione regolare, ma prima viene inserito il riempimento zero tra tutti i pixel nell'input. Questo è più facile da vedere con un'immagine: <<transp_conv>> mostra un diagramma tratto dall'eccellente [articolo sull'aritmetica convoluzionale](https://arxiv.org/abs/1603.07285) di cui abbiamo discusso in <<chapter_convolutions>>, che mostra una convoluzione trasposta 3×3 applicata a un'immagine 3×3.

<img alt="Una convoluzione trasposta" width="815" caption="Una convoluzione trasposta (per gentile concessione di Vincent Dumoulin e Francesco Visin)" id="transp_conv" src="images/att_00051.png">

Come vedi, il risultato è aumentare la dimensione dell'input. Puoi provarlo adesso utilizzando la classe `ConvLayer` di fastai; passa il parametro "transpose=True" per creare una convoluzione trasposta, invece di una normale, nella tua testa personalizzata.

Nessuno di questi approcci, tuttavia, funziona davvero bene. Il problema è che la nostra griglia 7×7 semplicemente non ha informazioni sufficienti per creare un output di 224×224 pixel. È richiesta un'enorme quantità di attivazioni di ciascuna di quelle celle della griglia per avere informazioni sufficienti per rigenerare completamente ogni pixel nell'output. La soluzione a questo problema è usare *salta connessioni*, come in ResNet, ma saltando dalle attivazioni nel corpo di ResNet fino alle attivazioni della convoluzione trasposta sul lato opposto dell'architettura. Questo approccio, illustrato in <<unet>>, è stato sviluppato da Olaf Ronneberger, Philipp Fischer e Thomas Brox nel documento del 2015 ["U-Net: Convolutional Networks for Biomedical Image Segmentation"](https://arxiv.org/abs/1505.04597). Sebbene l’articolo si concentri sulle applicazioni mediche, U-Net ha rivoluzionato tutti i tipi di modelli di visione generativa.

<img alt="L'architettura U-Net" width="630" caption="L'architettura U-Net (per gentile concessione di Olaf Ronneberger, Philipp Fischer e Thomas Brox)" id="unet" src="images/att_00052.png">

Questa immagine mostra il corpo della CNN sulla sinistra (in questo caso, è una normale CNN, non una ResNet, e stanno usando il pooling massimo 2×2 invece delle convoluzioni stride-2, poiché questo articolo è stato scritto prima dell'arrivo di ResNets) e i livelli convoluzionali trasposti ("up-conv") sulla destra. Quindi le connessioni saltate extra vengono visualizzate come frecce grigie che si incrociano da sinistra a destra (queste sono talvolta chiamate *connessioni incrociate*). Puoi capire perché si chiama "U-Net!"

Con questa architettura, l'input alle convoluzioni trasposte non è solo la griglia a bassa risoluzione nello strato precedente, ma anche la griglia a risoluzione più alta nella testa ResNet. Ciò consente a U-Net di utilizzare tutte le informazioni dell'immagine originale, quando necessario. Una sfida con U-Nets è che l'architettura esatta dipende dalla dimensione dell'immagine. fastai ha una classe "DynamicUnet" unica che genera automaticamente un'architettura della giusta dimensione in base ai dati forniti.

Concentriamoci ora su un esempio in cui sfruttiamo la libreria fastai per scrivere un modello personalizzato.

### Una rete siamese

In [ ]:
#hide
from fastai.vision.all import *
path = untar_data(URLs.PETS)
files = get_image_files(path/"images")

class SiameseImage(fastuple):
    def show(self, ctx=None, **kwargs): 
        img1,img2,same_breed = self
        if not isinstance(img1, Tensor):
            if img2.size != img1.size: img2 = img2.resize(img1.size)
            t1,t2 = tensor(img1),tensor(img2)
            t1,t2 = t1.permute(2,0,1),t2.permute(2,0,1)
        else: t1,t2 = img1,img2
        line = t1.new_zeros(t1.shape[0], t1.shape[1], 10)
        return show_image(torch.cat([t1,line,t2], dim=2), 
                          title=same_breed, ctx=ctx)
    
def label_func(fname):
    return re.match(r'^(.*)_\d+.jpg$', fname.name).groups()[0]

class SiameseTransform(Transform):
    def __init__(self, files, label_func, splits):
        self.labels = files.map(label_func).unique()
        self.lbl2files = {l: L(f for f in files if label_func(f) == l) for l in self.labels}
        self.label_func = label_func
        self.valid = {f: self._draw(f) for f in files[splits[1]]}
        
    def encodes(self, f):
        f2,t = self.valid.get(f, self._draw(f))
        img1,img2 = PILImage.create(f),PILImage.create(f2)
        return SiameseImage(img1, img2, t)
    
    def _draw(self, f):
        same = random.random() < 0.5
        cls = self.label_func(f)
        if not same: cls = random.choice(L(l for l in self.labels if l != cls)) 
        return random.choice(self.lbl2files[cls]),same
    
splits = RandomSplitter()(files)
tfm = SiameseTransform(files, label_func, splits)
tls = TfmdLists(files, tfm, splits=splits)
dls = tls.dataloaders(after_item=[Resize(224), ToTensor], 
    after_batch=[IntToFloatTensor, Normalize.from_stats(*imagenet_stats)])

Torniamo alla pipeline di input che abbiamo configurato in <<chapter_midlevel_data>> per una rete siamese. Se ricordi, consisteva in una coppia di immagini con l'etichetta "Vero" o "Falso", a seconda che fossero nella stessa classe o meno.

Utilizzando ciò che abbiamo appena visto, creiamo un modello personalizzato per questa attività e addestriamolo. Come? Utilizzeremo un'architettura preaddestrata e passeremo attraverso di essa le nostre due immagini. Quindi possiamo concatenare i risultati e inviarli a un head personalizzato che restituirà due previsioni. In termini di moduli, questo assomiglia a questo:

In [ ]:
class SiameseModel(Module):
    def __init__(self, encoder, head):
        self.encoder,self.head = encoder,head
    
    def forward(self, x1, x2):
        ftrs = torch.cat([self.encoder(x1), self.encoder(x2)], dim=1)
        return self.head(ftrs)

Per creare il nostro codificatore, dobbiamo solo prendere un modello preallenato e tagliarlo, come abbiamo spiegato prima. La funzione `create_body` lo fa per noi; dobbiamo solo passarlo nel punto in cui vogliamo tagliare. Come abbiamo visto in precedenza, secondo il dizionario dei metadati per i modelli pre-addestrati, il valore di taglio per un resnet è "-2":

In [ ]:
encoder = create_body(resnet34, cut=-2)

Quindi possiamo creare la nostra testa. Uno sguardo al codificatore ci dice che l'ultimo livello ha 512 funzioni, quindi questa testa dovrà ricevere "512*2". Perché 2? Dobbiamo moltiplicare per 2 perché abbiamo due immagini. Quindi creiamo la testa come segue:

In [ ]:
head = create_head(512*2, 2, ps=0.5)

Con il nostro encoder e la testa, ora possiamo costruire il nostro modello:

In [ ]:
model = SiameseModel(encoder, head)

Prima di utilizzare "Learner", abbiamo altre due cose da definire. Innanzitutto dobbiamo definire la funzione di perdita che vogliamo utilizzare. Si tratta di entropia incrociata regolare, ma poiché i nostri obiettivi sono booleani, dobbiamo convertirli in numeri interi altrimenti PyTorch genererà un errore:

In [ ]:
def loss_func(out, targ):
    return nn.CrossEntropyLoss()(out, targ.long())

Ancora più importante, per sfruttare appieno il trasferimento di apprendimento, dobbiamo definire uno *splitter* personalizzato. Uno splitter è una funzione che indica alla libreria fastai come dividere il modello in gruppi di parametri. Questi vengono utilizzati dietro le quinte per addestrare solo la testa di un modello quando trasferiamo l'apprendimento. 

Qui vogliamo due gruppi di parametri: uno per l'encoder e uno per la testa. Possiamo quindi definire il seguente splitter (`params` è semplicemente una funzione che restituisce tutti i parametri di un dato modulo):

In [ ]:
def siamese_splitter(model):
    return [params(model.encoder), params(model.head)]

Quindi possiamo definire il nostro "Studente" passando i dati, il modello, la funzione di perdita, lo splitter e qualsiasi metrica che desideriamo. Poiché non stiamo utilizzando una funzione comoda di fastai per il trasferimento dell'apprendimento (come `vision_learner`), dobbiamo chiamare manualmente `learn.freeze`. Ciò assicurerà che solo l'ultimo gruppo di parametri (in questo caso, la testa) venga addestrato:

In [ ]:
learn = Learner(dls, model, loss_func=loss_func, 
                splitter=siamese_splitter, metrics=accuracy)
learn.freeze()

Quindi possiamo addestrare direttamente il nostro modello con i soliti metodi:

In [ ]:
learn.fit_one_cycle(4, 3e-3)

epoch,train_loss,valid_loss,accuracy,time
0,0.367015,0.281242,0.885656,00:26
1,0.307688,0.214721,0.915426,00:26
2,0.275221,0.170615,0.936401,00:26
3,0.223771,0.159633,0.943843,00:26


Prima di sbloccare e perfezionare ulteriormente l'intero modello con tassi di apprendimento discriminativi (ovvero: un tasso di apprendimento inferiore per il corpo e uno più alto per la testa):

In [ ]:
learn.unfreeze()
learn.fit_one_cycle(4, slice(1e-6,1e-4))

epoch,train_loss,valid_loss,accuracy,time
0,0.212744,0.159033,0.944520,00:35
1,0.201893,0.159615,0.942490,00:35
2,0.204606,0.152338,0.945196,00:36
3,0.213203,0.148346,0.947903,00:36


Il 94,8\% è molto buono se ricordiamo che un classificatore addestrato allo stesso modo (senza aumento dei dati) aveva un tasso di errore del 7%.

Ora che abbiamo visto come creare modelli completi di visione artificiale all'avanguardia, passiamo alla PNL.

## Elaborazione del linguaggio naturale

Convertire un modello linguistico AWD-LSTM in un classificatore di trasferimento di apprendimento, come abbiamo fatto in <<chapter_nlp>>, segue un processo molto simile a quello che abbiamo fatto con `vision_learner` nella prima sezione di questo capitolo. In questo caso non abbiamo bisogno di un "meta" dizionario, perché non abbiamo una tale varietà di architetture da supportare nel corpo. Tutto quello che dobbiamo fare è selezionare l'RNN in stack per l'encoder nel modello linguistico, che è un singolo modulo PyTorch. Questo codificatore fornirà un'attivazione per ogni parola dell'input, poiché un modello linguistico deve produrre una previsione per ogni parola successiva.

Per creare un classificatore da questo utilizziamo un approccio descritto nel [documento ULMFiT](https://arxiv.org/abs/1801.06146) come "BPTT for Text Classification (BPT3C)":

> : Dividiamo il documento in lotti di lunghezza fissa di dimensione *b*. All'inizio di ogni lotto il modello viene inizializzato con lo stato finale del lotto precedente; teniamo traccia degli stati nascosti per il pooling medio e massimo; i gradienti vengono propagati all'indietro ai lotti i cui stati nascosti hanno contribuito alla previsione finale. In pratica utilizziamo sequenze di backpropagation di lunghezza variabile.

In altre parole, il classificatore contiene un ciclo "for", che esegue il ciclo su ogni batch di una sequenza. Lo stato viene mantenuto tra i batch e le attivazioni di ciascun batch vengono archiviate. Alla fine, utilizziamo lo stesso trucco di pooling concatenato medio e massimo che utilizziamo per i modelli di visione artificiale, ma questa volta non eseguiamo il pooling su celle della griglia CNN, ma su sequenze RNN.

Per questo ciclo `for` dobbiamo raccogliere i nostri dati in batch, ma ogni testo deve essere trattato separatamente, poiché ognuno ha le proprie etichette. Tuttavia, è molto probabile che questi testi non avranno tutti la stessa lunghezza, il che significa che non potremo metterli tutti nello stesso array, come abbiamo fatto con il modello linguistico.

È qui che il riempimento ci aiuta: quando prendiamo un gruppo di testi, determiniamo quello con la lunghezza maggiore, quindi riempiamo quelli più corti con un token speciale chiamato `xxpad`. Per evitare casi estremi in cui abbiamo un testo con 2.000 token nello stesso batch di un testo con 10 token (quindi molto riempimento e molti calcoli sprecati), alteriamo la casualità assicurandoci che testi di dimensioni comparabili siano messi insieme. I testi saranno ancora in un ordine alquanto casuale per il training set (per il validation set possiamo semplicemente ordinarli in ordine di lunghezza), ma non completamente.

Questo viene fatto automaticamente dietro le quinte dalla libreria fastai durante la creazione dei nostri `DataLoaders`.

## Tabellare

Infine, diamo un'occhiata ai modelli `fastai.tabular`. (Non abbiamo bisogno di esaminare separatamente il filtro collaborativo, poiché abbiamo già visto che questi modelli sono solo modelli tabulari o utilizzano l'approccio del prodotto scalare, che abbiamo implementato in precedenza da zero.)

Ecco il metodo "forward" per "TabularModel":

```pitone
se self.n_emb!= 0:
    x = [e(x_cat[:,i]) for i,e in enumerate(self.embeds)]
    x = torcia.gatto(x, 1)
    x = self.emb_drop(x)
se self.n_cont!= 0:
    x_cont = self.bn_cont(x_cont)
    x = torcia.cat([x, x_cont], 1) se self.n_emb != 0 altrimenti x_cont
restituire self.layers(x)
```

Non mostreremo `__init__` qui, poiché non è così interessante, ma esamineremo a turno ogni riga di codice in "forward". La prima riga:

```pitone
se self.n_emb!= 0:
```

sta solo verificando se ci sono incorporamenti di cui occuparsi: possiamo saltare questa sezione se abbiamo solo variabili continue. `self.embeds` contiene le matrici di incorporamento, quindi ottiene le attivazioni di ciascuna:
 
```pitone
    x = [e(x_cat[:,i]) for i,e in enumerate(self.embeds)]
```

e li concatena in un unico tensore:

```pitone
    x = torcia.gatto(x, 1)
```

Quindi viene applicato il dropout. Puoi passare `embd_p` a `__init__` per modificare questo valore:

```pitone
    x = self.emb_drop(x)
```

Ora testiamo se ci sono variabili continue da gestire:

```pitone
se self.n_cont!= 0:
```

Vengono passati attraverso un livello batchnorm:

```pitone
    x_cont = self.bn_cont(x_cont)
```

e concatenato con le attivazioni di incorporamento, se presenti:

```pitone
    x = torcia.cat([x, x_cont], 1) se self.n_emb != 0 altrimenti x_cont
```

Infine, questo viene passato attraverso i livelli lineari (ciascuno dei quali include batchnorm, se `use_bn` è `True`, e dropout, se `ps` è impostato su un valore o un elenco di valori):

```pitone
restituire self.layers(x)

```

Congratulazioni! Ora conosci ogni singolo pezzo delle architetture utilizzate nella libreria fastai!

## Concludendo le architetture

Come puoi vedere, i dettagli delle architetture di deep learning non devono spaventarti ora. Puoi guardare all'interno del codice di fastai e PyTorch e vedere cosa sta succedendo. Ancora più importante, cerca di capire *perché* sta succedendo. Dai un'occhiata ai documenti a cui viene fatto riferimento nel codice e prova a vedere come il codice corrisponde agli algoritmi descritti.

Ora che abbiamo analizzato tutte le parti di un modello e i dati che vengono trasmessi al suo interno, possiamo considerare cosa ciò significa per il deep learning pratico. Se disponi di dati illimitati, memoria illimitata e tempo illimitato, il consiglio è semplice: addestra un modello enorme su tutti i tuoi dati per un tempo davvero lungo. Ma il motivo per cui il deep learning non è semplice è perché i dati, la memoria e il tempo sono generalmente limitati. Se stai esaurendo la memoria o il tempo, la soluzione è addestrare un modello più piccolo. Se non sei in grado di allenarti abbastanza a lungo da sovraadattarti, non stai sfruttando la capacità del tuo modello.

Quindi, il primo passo è arrivare al punto in cui è possibile eseguire un overfit. Quindi la domanda è come ridurre questo overfitting. <<reduce_overfit>> mostra come consigliamo di dare priorità ai passaggi da lì.

<img alt="Procedura per ridurre l'overfitting" width="400" caption="Procedura per ridurre l'overfitting" id="reduce_overfit" src="images/att_00047.png">

Molti professionisti, di fronte a un modello di overfitting, iniziano esattamente dall’estremità sbagliata di questo diagramma. Il loro punto di partenza è utilizzare un modello più piccolo o una maggiore regolarizzazione. L'uso di un modello più piccolo dovrebbe essere assolutamente l'ultimo passo da compiere, a meno che l'addestramento del modello non richieda troppo tempo o memoria. Ridurre le dimensioni del modello riduce la capacità del modello di apprendere relazioni sottili nei dati.

Invece, il tuo primo passo dovrebbe essere quello di cercare di *creare più dati*. Ciò potrebbe comportare l'aggiunta di più etichette ai dati che già possiedi, la ricerca di attività aggiuntive che il tuo modello potrebbe dover risolvere (o, per vederla in un altro modo, l'identificazione di diversi tipi di etichette che potresti modellare) o la creazione di dati sintetici aggiuntivi utilizzando più o diverse tecniche di aumento dei dati. Grazie allo sviluppo di Mixup e di approcci simili, è ora disponibile un efficace aumento dei dati per quasi tutti i tipi di dati.

Una volta che hai tutti i dati che ritieni di poter ragionevolmente ottenere e li stai utilizzando nel modo più efficace possibile sfruttando tutte le etichette che puoi trovare e facendo tutto l'aumento che ha senso, se sei ancora in overfitting dovresti pensare a utilizzare architetture più generalizzabili. Ad esempio, l'aggiunta della normalizzazione batch può migliorare la generalizzazione.

Se sei ancora in fase di overfitting dopo aver fatto del tuo meglio nell'utilizzare i tuoi dati e nel mettere a punto la tua architettura, puoi dare un'occhiata alla regolarizzazione. In generale, l'aggiunta del dropout all'ultimo o due strati farà un buon lavoro nel regolarizzare il tuo modello. Tuttavia, come abbiamo appreso dalla storia dello sviluppo dell'AWD-LSTM, spesso accade che l'aggiunta di dropout di diverso tipo nel modello possa essere ancora più utile. In generale, un modello più grande con maggiore regolarizzazione è più flessibile e può quindi essere più accurato di un modello più piccolo con minore regolarizzazione.

Solo dopo aver considerato tutte queste opzioni ti consigliamo di provare a utilizzare una versione più piccola della tua architettura.

## Questionario

1. Qual è la "testa" di una rete neurale?
1. Qual è il "corpo" di una rete neurale?
1. Cos'è il "taglio" di una rete neurale? Perché dobbiamo farlo per il trasferimento dell’apprendimento?
1. Cos'è "model_meta"? Prova a stamparlo per vedere cosa c'è dentro.
1. Leggi il codice sorgente di "create_head" e assicurati di comprendere cosa fa ogni riga.
1. Guarda l'output di "create_head" e assicurati di capire perché è presente ogni livello e come la sorgente "create_head" lo ha creato.
1. Scopri come modificare il dropout, la dimensione del livello e il numero di livelli creati da "vision_learner" e vedi se riesci a trovare valori che si traducono in una migliore precisione dal riconoscimento dell'animale domestico.
1. Cosa fa `AdaptiveConcatPool2d`?
1. Cos'è l'"interpolazione del vicino più vicino"? Come può essere utilizzato per sovracampionare le attivazioni convoluzionali?
1. Cos'è una "convoluzione trasposta"? Qual è un altro nome per questo?
1. Crea un livello conv con "transpose=True" e applicalo a un'immagine. Controlla la forma dell'output.
1. Disegna l'architettura di U-Net.
1. Che cos'è "BPTT per la classificazione del testo" (BPT3C)?
1. Come gestiamo sequenze di diversa lunghezza in BPT3C?
1. Prova a eseguire ciascuna riga di `TabularModel.forward` separatamente, una riga per cella, in un taccuino e osserva le forme di input e output ad ogni passaggio.
1. Come viene definito "self.layers" in "TabularModel"?
1. Quali sono i cinque passaggi per prevenire l’adattamento eccessivo?
1. Perché non riduciamo la complessità dell'architettura prima di provare altri approcci per prevenire l'overfitting?

### Ulteriori ricerche

1. Scrivi la tua testa personalizzata e prova ad addestrare il riconoscimento dell'animale domestico con essa. Vedi se riesci a ottenere un risultato migliore rispetto a quello predefinito di fastai.
1. Prova a passare da "AdaptiveConcatPool2d" a "AdaptiveAvgPool2d" in una testata CNN e osserva la differenza.
1. Scrivi il tuo splitter personalizzato per creare un gruppo di parametri separato per ogni blocco ResNet e un gruppo separato per lo stelo. Prova ad allenarti con esso e vedi se migliora il riconoscimento degli animali domestici.
1. Leggi il capitolo online sui modelli di immagine generativa e crea il tuo colorizzatore, modello a super risoluzione o modello di trasferimento di stile.
1. Crea una testa personalizzata utilizzando l'interpolazione del vicino più vicino e usala per eseguire la segmentazione su CamVid.